# Navigation — Deep Q-Network (DQN)

Train an agent to collect yellow bananas (and avoid blue ones) in Unity's Banana Collector environment.

The environment is considered **solved** when the agent gets an average score of **+13 over 100 consecutive episodes**.

> **Workspace note:** Enable **GPU** for training. The Banana environment file path differs between the Udacity Workspace (`/data/...`) and a local download (`Banana.app`, `Banana_Linux/Banana.x86_64`, etc.) — adjust `file_name` in the cell below.

## 1. Start the Environment

In [ ]:
from unityagents import UnityEnvironment
import numpy as np

# Udacity Workspace path:
env = UnityEnvironment(file_name="/data/Banana_Linux_NoVis/Banana.x86_64")
# Local (Mac) example:
# env = UnityEnvironment(file_name="Banana.app")

# get the default brain
brain_name = env.brain_names[0]
brain = env.brains[brain_name]

## 2. Examine the State and Action Spaces

In [ ]:
env_info = env.reset(train_mode=True)[brain_name]

action_size = brain.vector_action_space_size          # expected: 4
state = env_info.vector_observations[0]
state_size = len(state)                                # expected: 37

print('Number of agents:', len(env_info.agents))
print('Number of actions:', action_size)
print('States have length:', state_size)
print('Example state:', state)

## 3. Train the Agent with DQN

In [ ]:
from collections import deque
import torch
import matplotlib.pyplot as plt
%matplotlib inline

from dqn_agent import Agent

agent = Agent(state_size=state_size, action_size=action_size, seed=0)


def dqn(n_episodes=2000, max_t=1000, eps_start=1.0, eps_end=0.01, eps_decay=0.995):
    """Deep Q-Learning training loop."""
    scores = []                          # list of scores from each episode
    scores_window = deque(maxlen=100)    # last 100 scores
    eps = eps_start
    for i_episode in range(1, n_episodes + 1):
        env_info = env.reset(train_mode=True)[brain_name]
        state = env_info.vector_observations[0]
        score = 0
        for t in range(max_t):
            action = int(agent.act(state, eps))
            env_info = env.step(action)[brain_name]
            next_state = env_info.vector_observations[0]
            reward = env_info.rewards[0]
            done = env_info.local_done[0]
            agent.step(state, action, reward, next_state, done)
            state = next_state
            score += reward
            if done:
                break
        scores_window.append(score)
        scores.append(score)
        eps = max(eps_end, eps_decay * eps)
        print('\rEpisode {}\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_window)), end="")
        if i_episode % 100 == 0:
            print('\rEpisode {}\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_window)))
        if np.mean(scores_window) >= 13.0:
            print('\nEnvironment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(
                i_episode - 100, np.mean(scores_window)))
            torch.save(agent.qnetwork_local.state_dict(), 'checkpoint.pth')
            break
    return scores


scores = dqn()

## 4. Plot the Scores

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
plt.plot(np.arange(len(scores)), scores)
plt.axhline(y=13.0, color='r', linestyle='--', label='solved (+13)')
plt.ylabel('Score')
plt.xlabel('Episode #')
plt.legend()
plt.savefig('assets/scores.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Watch a Trained Agent

In [ ]:
agent.qnetwork_local.load_state_dict(torch.load('checkpoint.pth'))

env_info = env.reset(train_mode=False)[brain_name]
state = env_info.vector_observations[0]
score = 0
while True:
    action = int(agent.act(state, eps=0.0))   # greedy
    env_info = env.step(action)[brain_name]
    state = env_info.vector_observations[0]
    score += env_info.rewards[0]
    if env_info.local_done[0]:
        break
print('Score: {}'.format(score))

In [ ]:
env.close()